Chargement des bibliothèques

In [1]:
import pandas as pd
import numpy as np

Chargement des données

In [2]:
sinistre = pd.read_csv('Sinistre.csv', sep=';', low_memory = False)

In [3]:
sinistre.head(5)

,idx_sin,gar_sin,surv_sin,decl_sin,clo_sin,gest_sin,mt_eval,mt_regl
0,S18-0043146,AssVHR,2018-12-08,2018-12-10,NaN,2018-12-10,170.00,0.00
1,S18-0043146,AssVHR,2018-12-08,2018-12-10,NaN,2018-12-31,170.00,0.00
2,S18-0043146,AssVHR,2018-12-08,2018-12-10,2019-08-05,2019-08-05,262.86,262.86
3,S18-0057961,Ass0km,2018-10-30,2018-10-30,NaN,2018-10-30,150.00,0.00
4,S18-0057961,Ass0km,2018-10-30,2018-10-30,NaN,2018-12-31,150.00,0.00


Nettoyage des données qualitatives

In [4]:
sinistre1 = sinistre.copy()

# 2. Conversion de l'identifiant en texte (en préservant les vrais NaN)
# 'idx_sin' est l'identifiant du sinistre
if 'idx_sin' in sinistre1.columns:
    sinistre1['idx_sin'] = sinistre1['idx_sin'].astype(str).replace('nan', np.nan)

# 3. Conversion de la garantie en catégorie
# 'gar_sin' représente la garantie actionnée (ex: BDG, Vol, Responsabilité Civile)
if 'gar_sin' in sinistre1.columns:
    sinistre1['gar_sin'] = sinistre1['gar_sin'].astype('category')

In [5]:
# 1. Création de la copie sinistre2 à partir de sinistre1
sinistre2 = sinistre1.copy()

# 2. Dictionnaire de renommage
noms_colonnes_sin = {
    "idx_sin": "identifiant_sinistre",
    "gar_sin": "garantie_sinistre"
}

# 3. Application du renommage
sinistre2 = sinistre2.rename(columns=noms_colonnes_sin)

# 4. Vérification des nouveaux noms
print("Colonnes de sinistre2 après renommage :")
print(sinistre2.columns.tolist())

# Petit check sur les premières lignes
print("\n Aperçu des données :")
print(sinistre2[['identifiant_sinistre', 'garantie_sinistre']].head())

Colonnes de sinistre2 après renommage :
['identifiant_sinistre', 'garantie_sinistre', 'surv_sin', 'decl_sin', 'clo_sin', 'gest_sin', 'mt_eval', 'mt_regl']

 Aperçu des données :
  identifiant_sinistre garantie_sinistre
0          S18-0043146            AssVHR
1          S18-0043146            AssVHR
2          S18-0043146            AssVHR
3          S18-0057961            Ass0km
4          S18-0057961            Ass0km


In [6]:
print("Liste des garanties présentes :")
print(sinistre2['garantie_sinistre'].unique())

Liste des garanties présentes :
['AssVHR', 'Ass0km', 'AssBase']
Categories (3, object): ['Ass0km', 'AssBase', 'AssVHR']


In [7]:
# 1. Création de la copie sinistre3
sinistre3 = sinistre2.copy()

# 2. Conversion temporaire en string pour modifier les libellés librement
sinistre3['garantie_sinistre'] = sinistre3['garantie_sinistre'].astype(str)

# 3. Dictionnaire de correspondance
mapping_garanties = {
    'Ass0km': 'Assistance 0 km',
    'AssBase': 'Assistance de base',
    'AssVHR': 'Assistance véhicule de remplacement'
}

# 4. Application du remplacement
sinistre3['garantie_sinistre'] = sinistre3['garantie_sinistre'].replace(mapping_garanties)

# 5. Conversion finale en 'category' pour l'optimisation
sinistre3['garantie_sinistre'] = sinistre3['garantie_sinistre'].astype('category')

# Vérification du résultat
print("Nouvelles modalités de la garantie :")
print(sinistre3['garantie_sinistre'].unique())

print("\nRépartition mise à jour :")
print(sinistre3['garantie_sinistre'].value_counts())

Nouvelles modalités de la garantie :
['Assistance véhicule de remplacement', 'Assistance 0 km', 'Assistance de base']
Categories (3, object): ['Assistance 0 km', 'Assistance de base', 'Assistance véhicule de remplacement']

Répartition mise à jour :
garantie_sinistre
Assistance de base                     57685
Assistance 0 km                         8575
Assistance véhicule de remplacement     5870
Name: count, dtype: int64


Nettoyage des données quantitatives

In [8]:
# 1. Création de la copie
sinistre4 = sinistre3.copy()

# 2. Conversion en numérique (float)
# errors='coerce' transformera les textes incohérents en NaN
sinistre4['mt_eval'] = pd.to_numeric(sinistre4['mt_eval'], errors='coerce')
sinistre4['mt_regl'] = pd.to_numeric(sinistre4['mt_regl'], errors='coerce')

# 3. Renommage en français pour la clarté
noms_montants = {
    "mt_eval": "montant_evaluation",
    "mt_regl": "montant_reglement"
}
sinistre4 = sinistre4.rename(columns=noms_montants)

# 4. Traitement des aberrations
# Un montant de sinistre ne peut pas être négatif (sauf cas très rare de recours, 
# mais ici on va considérer que < 0 est une erreur de saisie)
sinistre4.loc[sinistre4['montant_evaluation'] < 0, 'montant_evaluation'] = np.nan
sinistre4.loc[sinistre4['montant_reglement'] < 0, 'montant_reglement'] = np.nan

# 5. Vérification statistique
print("Résumé des montants de sinistres :")
print(sinistre4[['montant_evaluation', 'montant_reglement']].describe())

Résumé des montants de sinistres :
       montant_evaluation  montant_reglement
count        70969.000000       70969.000000
mean           262.693986         164.312627
std            635.553220         665.228942
min              0.000000           0.000000
25%            170.000000           0.000000
50%            187.070000           0.000000
75%            215.000000         192.310000
max           7429.160000        7429.160000


Nettoyage des données temporelles

In [12]:
# 1. Création de la copie sinistre5
sinistre5 = sinistre4.copy()

# 2. Dictionnaire de renommage pour plus de clarté
noms_temporels = {
    "surv_sin": "date_survenance_sinistre",
    "decl_sin": "date_declaration_sinistre",
    "clo_sin": "date_cloture_sinistre"
}

sinistre5 = sinistre5.rename(columns=noms_temporels)

# 3. Conversion en format datetime
# errors='coerce' transforme les dates illisibles en NaT (Not a Time)
date_cols = ["date_survenance_sinistre", "date_declaration_sinistre", "date_cloture_sinistre"]

for col in date_cols:
    sinistre5[col] = pd.to_datetime(sinistre5[col], errors='coerce')

# 4. Vérification de la cohérence temporelle
# Un dossier ne devrait pas être clôturé avant d'avoir été déclaré
# On crée une alerte pour les lignes incohérentes
incoherences = sinistre5[sinistre5['date_cloture_sinistre'] < sinistre5['date_declaration_sinistre']]
print(f"Nombre de dossiers avec clôture avant déclaration : {len(incoherences)}")

# 5. Affichage du résumé
print("\nRésumé des variables temporelles :")
print(sinistre5[date_cols].describe)

Nombre de dossiers avec clôture avant déclaration : 0

Résumé des variables temporelles :
<bound method NDFrame.describe of       date_survenance_sinistre date_declaration_sinistre date_cloture_sinistre
0                   2018-12-08                2018-12-10                   NaT
1                   2018-12-08                2018-12-10                   NaT
2                   2018-12-08                2018-12-10            2019-08-05
3                   2018-10-30                2018-10-30                   NaT
4                   2018-10-30                2018-10-30                   NaT
...                        ...                       ...                   ...
72125               2023-02-24                2023-02-25            2023-03-16
72126               2023-10-01                2023-10-02                   NaT
72127               2023-10-01                2023-10-02            2023-11-24
72128               2023-12-06                2023-12-07                   NaT
72129  

Traitement des doublons

In [13]:
# 1. Création de la copie sinistre6
sinistre6 = sinistre5.copy()

# 2. Comptage avant suppression pour ton rapport de nettoyage
nb_lignes_avant = len(sinistre6)

# 3. Suppression des doublons stricts (lignes parfaitement identiques)
sinistre6 = sinistre6.drop_duplicates(keep='first')

# 4. Calcul du nombre de doublons supprimés
nb_doublons_sin = nb_lignes_avant - len(sinistre6)

print(f"Nombre de doublons stricts supprimés dans la table Sinistre : {nb_doublons_sin}")
print(f"Nombre de lignes finales dans sinistre6 : {len(sinistre6)}")

# 5. Vérification rapide : un identifiant_sinistre est-il toujours présent plusieurs fois ?
doublons_id = sinistre6['identifiant_sinistre'].duplicated().sum()
if doublons_id > 0:
    print(f"\nAttention : Il reste {doublons_id} identifiants de sinistres présents sur plusieurs lignes.")
    print("Cela peut arriver si un sinistre a plusieurs garanties ou plusieurs règlements.")

Nombre de doublons stricts supprimés dans la table Sinistre : 68
Nombre de lignes finales dans sinistre6 : 72062

Attention : Il reste 36848 identifiants de sinistres présents sur plusieurs lignes.
Cela peut arriver si un sinistre a plusieurs garanties ou plusieurs règlements.


Traitement des valeurs manquantes

In [14]:
# 1. Calcul du nombre et du pourcentage de valeurs manquantes par colonne
missing_sin = sinistre6.isnull().sum()
missing_sin_pct = (sinistre6.isnull().sum() / len(sinistre6)) * 100

# 2. Création du tableau récapitulatif
df_missing_sin = pd.DataFrame({
    'Valeurs Manquantes': missing_sin,
    'Pourcentage (%)': missing_sin_pct.round(2)
})

# 3. Affichage des colonnes qui ont des manquants (triées)
print("État des lieux des valeurs manquantes dans sinistre6 :")
print(df_missing_sin[df_missing_sin['Valeurs Manquantes'] > 0].sort_values(by='Valeurs Manquantes', ascending=False))

État des lieux des valeurs manquantes dans sinistre6 :
                       Valeurs Manquantes  Pourcentage (%)
date_cloture_sinistre               39031            54.16
montant_evaluation                   1161             1.61
montant_reglement                    1161             1.61


Le fait que montant_evaluation et montant_reglement aient exactement le même nombre de manquants (1 161) suggère qu'il s'agit des mêmes dossiers. Ce sont probablement des sinistres qui viennent juste d'être déclarés (flux "frais"). L'expert n'a pas encore rendu son rapport (pas d'évaluation) et la compagnie n'a donc encore rien payé (pas de règlement).